<a href="https://colab.research.google.com/github/alifah1940/Cyberbullying-and-Emotion-Analysis-System/blob/main/fyp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Required Libraries

In [ ]:
!pip install datasets pandas nltk scikit-learn matplotlib seaborn

# Import Libraries



In [ ]:
import re
import string
import pandas as pd
import numpy as np
import nltk
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from datasets import load_dataset
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

print("All libraries imported successfully.")

In [ ]:
print("\nLoading MYBully dataset from Hugging Face...")

# Load the MYBully dataset
dataset = load_dataset("mohanrj/MYBully")

# Convert the 'train' split to a Pandas DataFrame for easier manipulation
df = pd.DataFrame(dataset['train'])

# Explore dataset
print(f"Dataset loaded! Shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")

# Show label distribution
print(f"\nLabel distribution:")
print(df['Bully'].value_counts())

# Preview raw data (first 8 rows)
print("\nSample raw tweets:")
display(df[['Tweet', 'Bully']].head(8)) # Changed to use display() for better table formatting

# Load the MYBully Dataset from Hugging Face

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='Emotion', hue='Emotion', palette='viridis', order=df['Emotion'].value_counts().index, legend=False)
plt.title('Distribution of Emotions in the Dataset')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(data=df, x='Bully', hue='Bully', palette='coolwarm', legend=False)
plt.title('Distribution of Bully Labels')
plt.xlabel('Bully Label')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

### Correlation between 'Emotion' and 'Bully' categories
to analyze the relationship between the 'Emotion' and 'Bully' categories

In [ ]:

crosstab_emotion_bully = pd.crosstab(df['Emotion'], df['Bully'])
display(crosstab_emotion_bully)

plt.figure(figsize=(10, 7))
sns.heatmap(crosstab_emotion_bully, annot=True, fmt='d', cmap='YlGnBu', linewidths=.5)
plt.title('Correlation between Emotion and Bully Categories')
plt.xlabel('Bully Label')
plt.ylabel('Emotion')
plt.show()

In [ ]:
emotion_df = df[df['Emotion'].isin([
    'Anger',
    'Neutral',
    'Happiness',
    'Sadness'
])]

emotion_df['Emotion'].value_counts()

# Define Stopwords
Combines Malay + English stopwords, but KEEPS slang/bully-related words

In [ ]:
from nltk.corpus import stopwords

In [ ]:
# Standard English stopwords
english_stopwords = set(stopwords.words('english'))

In [ ]:
# Common Malay stopwords
malay_stopwords = {
    'yang', 'dan', 'di', 'ke', 'dari', 'pada', 'ini', 'itu', 'dengan',
    'untuk', 'tidak', 'adalah', 'ada', 'juga', 'atau', 'oleh', 'saya',
    'aku', 'kamu', 'dia', 'mereka', 'kami', 'kita', 'akan', 'sudah',
    'telah', 'sudah', 'pun', 'lah', 'kan', 'je', 'je', 'tapi', 'kalau',
    'bila', 'sebab', 'macam', 'bukan', 'lebih', 'semua', 'satu', 'dua',
    'dalam', 'lagi', 'sini', 'sana', 'tu', 'ni', 'ha', 'ah', 'oh',
    'ya', 'ye', 'la', 'eh', 'wei', 'weh', 'bagi', 'boleh', 'tak',
    'nak', 'pon', 'ngan', 'sama', 'dah', 'bila', 'mana', 'masa',
    'bila', 'hari', 'saya', 'awak', 'dia', 'sana', 'sini', 'itu',
    'ini', 'siapa', 'apa', 'bila', 'kenapa', 'macam', 'punya', 'tau'
}

# bully-related slang
keep_words = {
    'bodoh', 'bangang', 'stupid', 'babi', 'celaka', 'sial', 'gila',
    'hina', 'maki', 'kutuk', 'attack', 'bully', 'teruk', 'jahat',
    'buruk', 'hodoh', 'pemalas', 'lembab', 'bengap', 'bahlol',
    'reject', 'loser', 'ugly', 'fat', 'hate', 'kill', 'die',
    'useless', 'pathetic', 'idiot', 'fool', 'dumb', 'moron'
}

In [ ]:
# Combine all stopwords, then subtract the ones we want to keep
all_stopwords = (english_stopwords | malay_stopwords) - keep_words

print(f"\nTotal stopwords defined: {len(all_stopwords)}")
print(f"Protected bully-related slang words: {len(keep_words)}")

# Define Text Preprocessing Function

In [ ]:
def preprocess_text(text):
    """
    Cleans a raw tweet through the following steps:
    1. Lowercase all characters
    2. Remove URLs (http/https/www links)
    3. Remove @mentions (e.g. @username)
    4. Remove hashtag symbols (keep the word)
    5. Remove emojis and non-ASCII characters
    6. Remove punctuation and special characters
    7. Tokenize into individual words
    8. Remove stopwords (but keep slang/bully words)
    9. Rejoin tokens into a cleaned string

    Returns:
        cleaned_text (str): The processed tweet text
        tokens (list): List of individual tokens
    """

    # Guard against NaN or non-string values
    if not isinstance(text, str):
        return "", []

    # Step 1: Lowercase
    text = text.lower()

    # Step 2: Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # Step 3: Remove @mentions
    text = re.sub(r'@\w+', '', text)

    # Step 4: Remove hashtag symbol but keep the word
    text = re.sub(r'#', '', text)

    # Step 5: Remove emojis and non-ASCII characters
    text = text.encode('ascii', 'ignore').decode('ascii')

    # Step 6: Remove punctuation and special characters (keep letters and spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # Step 7: Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Step 8: Tokenize
    tokens = word_tokenize(text)

    # Step 9: Remove stopwords (preserve words in keep_words set)
    tokens = [
        word for word in tokens
        if word not in all_stopwords or word in keep_words
    ]

    # Step 10: Remove very short tokens
    tokens = [word for word in tokens if len(word) > 1]

    # Rejoin tokens into a string
    cleaned_text = ' '.join(tokens)

    return cleaned_text, tokens

print("Preprocessing function defined.")


# Apply Preprocessing to the Entire Dataset

In [ ]:
print("\nPreprocessing all tweets...")

# Apply the function to every row
results = df['Tweet'].apply(preprocess_text)

# Store cleaned text and tokens as new columns
df['cleaned_text'] = results.apply(lambda x: x[0])
df['tokens'] = results.apply(lambda x: x[1])

print(f"Preprocessing complete! {len(df)} tweets processed.")

# Raw vs. Cleaned Text Examples

In [ ]:
print("\n" + "="*70)
print("Raw vs. Cleaned Text (Sample)")
print("="*70)

# Select 5 bully and 5 non-bully examples
# Fix: Use 'df' instead of the undefined 'df_clean'
sample_indices = pd.concat([
    df[df['Bully'] == 'Yes'].head(5),
    df[df['Bully'] == 'No'].head(5)
]).index

for i, idx in enumerate(sample_indices):
    print(f"\n--- Example {i+1} ---")
    print(f"  RAW TEXT   : {df.loc[idx, 'Tweet']}")
    print(f"  CLEANED    : {df.loc[idx, 'cleaned_text']}")
    print(f"  LABEL      : {'Bully' if df.loc[idx, 'Bully'] == 'Yes' else 'Non-Bully'}")

# Show Tokenized Text

In [ ]:
print("\n" + "="*70)
print("OUTPUT 2: Tokenized Text Examples")
print("="*70)

for i, idx in enumerate(sample_indices):
    print(f"\n--- Example {i+1} ---")
    print(f"  CLEANED TEXT : {df.loc[idx, 'cleaned_text']}")
    print(f"  TOKENS       : {df.loc[idx, 'tokens']}")

# TF-IDF Feature Extraction

In [ ]:
print("\n" + "="*70)
print("OUTPUT 3: TF-IDF Feature Matrix")
print("="*70)

# PROCESS TF-IDF (Feature Extraction)
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Using 'df' instead of 'data_final' and 'cleaned_text' instead of 'Translated_Review_MS'
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['cleaned_text'].astype('U'))

tf = TfidfVectorizer()
text_tf = tf.fit_transform(df['cleaned_text'].astype('U'))

print(f"Feature Matrix Shape: {text_tf.shape}")
print("\nSparse Matrix Preview:")
print(text_tf)

# Extract Top Bullying-Related Words


In [ ]:
print("\n" + "="*70)
print("OUTPUT 4: Top Bullying-Related Words by TF-IDF Score")
print("="*70)

# Get the boolean mask for bullying tweets from the *current* df
bully_mask = df['Bully'] == 'Yes'

# Get the integer *positions* (iloc) of these rows.
# This directly gives the row indices that correspond to text_tf.
bully_ilocs = np.where(bully_mask)[0]

# Extract TF-IDF scores for bullying tweets only using positional indices
bully_tfidf = text_tf[bully_ilocs]

# Compute mean score per word
mean_scores = np.asarray(bully_tfidf.mean(axis=0)).flatten()

# Fix: Get feature names from the 'tf' vectorizer defined in previous cell
feature_names = tf.get_feature_names_out()

# Create a sorted DataFrame of words and their scores
word_scores = pd.DataFrame({
    'word': feature_names,
    'mean_tfidf': mean_scores
}).sort_values('mean_tfidf', ascending=False)

# Show top 20 words
top_words = word_scores.head(20)
print("\nTop 20 Bullying-Related Words (by Mean TF-IDF Score):")
print(top_words.to_string(index=False))

# Visualization Bar Chart of Top Bullying Words

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')

fig, ax = plt.subplots(figsize=(12, 7))

# Color gradient: darker = higher score
colors = sns.color_palette("YlOrRd", n_colors=20)[::-1]

bars = ax.barh(
    top_words['word'],
    top_words['mean_tfidf'],
    color=colors,
    edgecolor='white',
    linewidth=0.5,
    height=0.7
)

# Add value labels on each bar
for bar, val in zip(bars, top_words['mean_tfidf']):
    ax.text(
        val + 0.0003,
        bar.get_y() + bar.get_height() / 2,
        f'{val:.4f}',
        va='center',
        ha='left',
        fontsize=9,
        color='#333333'
    )

# Formatting
ax.invert_yaxis()  # Highest score at the top
ax.set_xlabel('Mean TF-IDF Score', fontsize=13, labelpad=10)
ax.set_title(
    'Top 20 Bullying-Related Words\nby Mean TF-IDF Score (MYBully Dataset)',
    fontsize=15,
    fontweight='bold',
    pad=15
)
ax.set_xlim(0, top_words['mean_tfidf'].max() * 1.18)
ax.tick_params(axis='y', labelsize=11)
ax.tick_params(axis='x', labelsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add a note about the dataset
# Fix: Changed df_clean to df
fig.text(
    0.99, 0.01,
    f'Source: MYBully Dataset | n = {len(df)} tweets',
    ha='right', va='bottom',
    fontsize=8, color='grey',
    style='italic'
)

plt.tight_layout()
plt.savefig('tfidf_top_bullying_words.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved as 'tfidf_top_bullying_words.png'")

# Summary Statistics

In [ ]:
print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)

# Token length statistics
# Fix: Use 'df' instead of 'df_clean'
df['token_count'] = df['tokens'].apply(len)

print(f"\nDataset Size         : {len(df)} tweets")
print(f"Bully Tweets         : {(df['Bully'] == 'Yes').sum()} ({(df['Bully'] == 'Yes').mean()*100:.1f}%) ")
print(f"Non-Bully Tweets     : {(df['Bully'] == 'No').sum()} ({(df['Bully'] == 'No').mean()*100:.1f}%) ")
print(f"\nAvg Tokens (raw)     : {df['Tweet'].apply(lambda x: len(str(x).split())).mean():.1f}")
print(f"Avg Tokens (cleaned) : {df['token_count'].mean():.1f}")

# Fix: Use 'text_tf' which was defined in the TF-IDF extraction cell
print(f"TF-IDF Matrix Shape  : {text_tf.shape[0]} rows × {text_tf.shape[1]} features")

# MODEL TRAINING
SVM, LR, NB, KNN, DECISION TREE, SNN, CNN, LSTM, mBERT

In [ ]:
import warnings
import torch

# Sklearn — classical ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix,
                             classification_report)
from sklearn.preprocessing import MinMaxScaler

# Keras / TensorFlow — Deep Learning
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Dense, Dropout, Embedding, Conv1D,
                                      GlobalMaxPooling1D, LSTM as KerasLSTM,
                                      Input, Flatten)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

# Hugging Face — BERT
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from transformers import DataCollatorWithPadding

warnings.filterwarnings('ignore')
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("✅ All libraries loaded.")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"ኒ Device: {DEVICE}")

to make evaluation process easier and consistent

In [ ]:
RESULTS = {}

def evaluate_model(name, y_true, y_pred):
    """Compute and store metrics; print classification report."""
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    RESULTS[name] = {'Accuracy': acc, 'Precision': prec,
                     'Recall': rec, 'F1-Score': f1}
    print(f"\n{'='*55}")
    print(f"  MODEL: {name}")
    print(f"{'='*55}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"\n{classification_report(y_true, y_pred, target_names=['Non-Bully','Bully'])}")
    return acc, prec, rec, f1

def plot_confusion_matrix(name, y_true, y_pred):
    """Plot a styled confusion matrix and save it."""
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Non-Bully','Bully'],
                yticklabels=['Non-Bully','Bully'],
                linewidths=0.5, linecolor='white',
                annot_kws={"size": 13, "weight": "bold"})
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label', fontsize=11)
    ax.set_title(f'Confusion Matrix — {name}', fontsize=13, fontweight='bold', pad=12)
    plt.tight_layout()
    fname = f"cm_{name.replace(' ','_').lower()}.png"
    plt.savefig(fname, dpi=130, bbox_inches='tight')
    plt.show()
   # print(f"  💾 Saved: {fname}")

print("✅ Utility functions ready.")


#TF-IDF Features (for Classical ML Models)

In [ ]:
# Fix: Use 'cleaned_text' and map labels to numeric
bully_tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2),
                         min_df=2, sublinear_tf=True)

# 1. Use correct column name: 'cleaned_text'
X_tfidf = bully_tfidf_vectorizer.fit_transform(df['cleaned_text'])

# 2. Map 'Bully' column labels ('Yes'/'No') to numeric (1/0)
y = df['Bully'].map({'Yes': 1, 'No': 0}).values

# Scale to [0,1] for Naive Bayes
scaler = MinMaxScaler()
X_tfidf_dense = scaler.fit_transform(X_tfidf.toarray())

X_tr, X_te, y_tr, y_te = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y)
X_tr_d, X_te_d, _, _ = train_test_split(
    X_tfidf_dense, y, test_size=0.2, random_state=42, stratify=y)

print(f"✅ TF-IDF split → Train: {X_tr.shape[0]} | Test: {X_te.shape[0]}")

#SVM

In [ ]:
print("\nSVM...")
svm = SVC(kernel='linear', C=1.0, random_state=42)
svm.fit(X_tr, y_tr)
y_pred_svm = svm.predict(X_te)
evaluate_model("SVM", y_te, y_pred_svm)
plot_confusion_matrix("SVM", y_te, y_pred_svm)

#LR

In [ ]:
print("\nLogistic Regression...")
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_tr, y_tr)
y_pred_lr = lr.predict(X_te)
evaluate_model("Logistic Regression", y_te, y_pred_lr)
plot_confusion_matrix("Logistic Regression", y_te, y_pred_lr)

#NB

In [ ]:
print("\n Naïve Bayes...")
nb = MultinomialNB(alpha=0.1)
nb.fit(X_tr_d, y_tr)          # Dense non-negative input
y_pred_nb = nb.predict(X_te_d)
evaluate_model("Naïve Bayes", y_te, y_pred_nb)
plot_confusion_matrix("Naïve Bayes", y_te, y_pred_nb)

#KNN

In [ ]:
print("\nKNN (k=7)...")
knn = KNeighborsClassifier(n_neighbors=7, metric='cosine', n_jobs=-1)
knn.fit(X_tr, y_tr)
y_pred_knn = knn.predict(X_te)
evaluate_model("KNN", y_te, y_pred_knn)
plot_confusion_matrix("KNN", y_te, y_pred_knn)

#DECISION TREE

In [ ]:
print("\nDecision Tree...")
dt = DecisionTreeClassifier(max_depth=20, random_state=42)
dt.fit(X_tr, y_tr)
y_pred_dt = dt.predict(X_te)
evaluate_model("Decision Tree", y_te, y_pred_dt)
plot_confusion_matrix("Decision Tree", y_te, y_pred_dt)


#Sequence Tokenization (for Deep Learning Models)

In [ ]:
MAX_WORDS  = 10000   # vocabulary size
MAX_LEN    = 50      # max sequence length (tweets are short)
EMBED_DIM  = 64      # embedding dimensions
BATCH_SIZE = 64

# Keras tokenizer
tok = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tok.fit_on_texts(df['cleaned_text'])
sequences = tok.texts_to_sequences(df['cleaned_text'])
X_seq = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

# Train/test split (same random state for reproducibility)
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(
    X_seq, y, test_size=0.2, random_state=42, stratify=y)

print(f"Sequence data split → Train: {Xs_tr.shape} | Test: {Xs_te.shape}")

#Helper — Train Keras DL Model

In [ ]:
def train_keras_model(model, X_train, y_train, X_val, y_val,
                       epochs=15, batch_size=BATCH_SIZE):
    """Compile, train, and return a Keras model with early stopping."""
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    es = EarlyStopping(monitor='val_loss', patience=3,
                       restore_best_weights=True, verbose=0)
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[es],
        verbose=1
    )
    return model, history

def predict_keras(model, X_test, threshold=0.5):
    """Return binary predictions from a Keras sigmoid model."""
    probs = model.predict(X_test, verbose=0).flatten()
    return (probs >= threshold).astype(int)

#print("✅ Keras helper functions ready.")

# SNN

In [ ]:
print("\nSimple Neural Network (SNN)...")

snn = Sequential([
    Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_LEN),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
], name='SNN')

snn.summary()
snn, snn_hist = train_keras_model(snn, Xs_tr, ys_tr, Xs_te, ys_te)

y_pred_snn = predict_keras(snn, Xs_te)
evaluate_model("Simple Neural Network", ys_te, y_pred_snn)
plot_confusion_matrix("Simple Neural Network", ys_te, y_pred_snn)

#CNN

In [ ]:
print("\nCNN...")

cnn = Sequential([
    Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_LEN),
    Conv1D(128, kernel_size=3, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(1, activation='sigmoid')
], name='CNN')

cnn.summary()
cnn, cnn_hist = train_keras_model(cnn, Xs_tr, ys_tr, Xs_te, ys_te)

y_pred_cnn = predict_keras(cnn, Xs_te)
evaluate_model("CNN", ys_te, y_pred_cnn)
plot_confusion_matrix("CNN", ys_te, y_pred_cnn)

#LTSM

EPOCH = 15

In [ ]:
print("\nLSTM...")

lstm_model = Sequential([
    Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_LEN),
    KerasLSTM(128, return_sequences=False, dropout=0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
], name='LSTM')

lstm_model.summary()
lstm_model, lstm_hist = train_keras_model(lstm_model, Xs_tr, ys_tr, Xs_te, ys_te)

y_pred_lstm = predict_keras(lstm_model, Xs_te)
evaluate_model("LSTM", ys_te, y_pred_lstm)
plot_confusion_matrix("LSTM", ys_te, y_pred_lstm)

#print("\n✅ All 3 deep learning models trained and evaluated.")

EPOCH = 10

In [ ]:
from tensorflow.keras.layers import Bidirectional

print("\n│ Building LSTM Model...")

model_lstm = Sequential([
    Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_LEN),
    Bidirectional(KerasLSTM(64, return_sequences=False)),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_lstm.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model_lstm.summary()

# Early stopping to prevent overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("\n│ Training LSTM...")
history = model_lstm.fit(
    Xs_tr, ys_tr,
    epochs=10,
    batch_size=BATCH_SIZE,
    validation_data=(Xs_te, ys_te),
    callbacks=[early_stop],
    verbose=1
)

# Evaluate
y_pred_lstm_prob = model_lstm.predict(Xs_te)
y_pred_lstm = (y_pred_lstm_prob > 0.5).astype(int).flatten()

evaluate_model("LSTM", ys_te, y_pred_lstm)
plot_confusion_matrix("LSTM", ys_te, y_pred_lstm)

# mBERT

In [ ]:
from torch.utils.data import Dataset
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# Ensure df exists
try:
    df
except NameError:
    print("Reloading dataset...")
    dataset = load_dataset("mohanrj/MYBully")
    df = pd.DataFrame(dataset['train'])

print("\nTraining BERT (mBERT)...")
print("ℹ  This will take 10–20 minutes on a Colab T4 GPU.")

BERT_MODEL_NAME = "bert-base-multilingual-cased"
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

X_raw = df['Tweet'].values
y_bert = df['Bully'].map({'Yes': 1, 'No': 0}).values

X_raw_tr, X_raw_te, y_bert_tr, y_bert_te = train_test_split(
    X_raw, y_bert, test_size=0.2, random_state=42, stratify=y_bert)

class BullyDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_ds = BullyDataset(X_raw_tr, y_bert_tr, bert_tokenizer)
test_ds  = BullyDataset(X_raw_te, y_bert_te, bert_tokenizer)

bert_model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL_NAME, num_labels=2)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
bert_model.to(DEVICE)

training_args = TrainingArguments(
    output_dir='./bert_output',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./bert_logs',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='no',
    load_best_model_at_end=False,
    fp16=(DEVICE == 'cuda'),
    report_to='none',
    disable_tqdm=True
)

data_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=data_collator
)

trainer.train()

preds_output = trainer.predict(test_ds)
y_pred_bert  = np.argmax(preds_output.predictions, axis=1)

evaluate_model("BERT (mBERT)", y_bert_te, y_pred_bert)
plot_confusion_matrix("BERT (mBERT)", y_bert_te, y_pred_bert)


#Comparison Bar Table

In [ ]:
print("\n" + "="*65)
print("SUMMARY : All Models Comparison")
print("="*65)

summary_df = pd.DataFrame(RESULTS).T.reset_index()
summary_df.columns = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score']
summary_df = summary_df.sort_values('F1-Score', ascending=False).reset_index(drop=True)
summary_df.index += 1  # Rank starts at 1

# Format to 4 decimal places for display
display_df = summary_df.copy()
for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    display_df[col] = display_df[col].map('{:.4f}'.format)

print(display_df.to_string())

# Save table as CSV
summary_df.to_csv('model_comparison.csv', index=False)
#print("\n💾 Saved: model_comparison.csv")

#Comparison Bar Chart

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
model_names = summary_df['Model'].tolist()
x = np.arange(len(model_names))
n_metrics = len(metrics)
bar_w = 0.18

palette = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

fig, ax = plt.subplots(figsize=(16, 6))

for i, (metric, color) in enumerate(zip(metrics, palette)):
    vals = summary_df[metric].values
    offset = (i - n_metrics / 2 + 0.5) * bar_w
    bars = ax.bar(x + offset, vals, width=bar_w, label=metric,
                  color=color, alpha=0.88, edgecolor='white', linewidth=0.6)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.2f}', ha='center', va='bottom', fontsize=7.5,
                fontweight='bold', color='#333333')

ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=25, ha='right', fontsize=10)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Comparison — Accuracy, Precision, Recall, F1-Score\n(MYBully Cyberbullying Dataset)',
             fontsize=14, fontweight='bold', pad=12)
ax.legend(loc='upper right', fontsize=10, framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('model_comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# EMOTION TRAINING

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# The dataset 'df' is already loaded from Hugging Face in a previous cell.
# No need to load MYBully.csv again.

# Filter only the 4 emotions used in this project
target_emotions = ["Anger", "Neutral", "Happiness", "Sadness"]
df = df[df['Emotion'].isin(target_emotions)]

# Features (text) and labels (emotion)
# The text column is named 'Tweet' in the loaded DataFrame, not 'Text'
X = df['Tweet']
y = df['Emotion']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF vectorization
emotion_tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = emotion_tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = emotion_tfidf_vectorizer.transform(X_test)

#LOGISTIC REGRESSION

In [ ]:
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_model.fit(X_train_tfidf, y_train)

lr_preds = lr_model.predict(X_test_tfidf)
print("Logistic Regression Results:")
print(classification_report(y_test, lr_preds))
print(confusion_matrix(y_test, lr_preds))

#EMOTION TRAINING - SUPPORT VECTOR MACHINE

In [ ]:
svm_model = SVC(kernel='linear', class_weight='balanced',probability=True )
svm_model.fit(X_train_tfidf, y_train)

svm_preds = svm_model.predict(X_test_tfidf)
print("SVM Results:")
print(classification_report(y_test, svm_preds))
print(confusion_matrix(y_test, svm_preds))

# Confusion Matrix Visualization (LR & SVM)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Function to plot confusion matrix
def plot_confusion_matrix(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred, labels=target_emotions)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_emotions,
                yticklabels=target_emotions)
    plt.title(f'Confusion Matrix - {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

# Plot for Logistic Regression
plot_confusion_matrix(y_test, lr_preds, "Logistic Regression")

# Plot for SVM
plot_confusion_matrix(y_test, svm_preds, "SVM")


#Bar Plot Visualization (Precision, Recall, F1‑Score)

In [ ]:
import numpy as np

# Function to plot metrics
def plot_metrics(y_true, y_pred, model_name):
    report = classification_report(y_true, y_pred, labels=target_emotions, output_dict=True)

    emotions = target_emotions
    precision = [report[e]['precision'] for e in emotions]
    recall = [report[e]['recall'] for e in emotions]
    f1 = [report[e]['f1-score'] for e in emotions]

    x = np.arange(len(emotions))
    width = 0.25

    plt.figure(figsize=(8,6))
    bars1 = plt.bar(x - width, precision, width, label='Precision', color='skyblue')
    bars2 = plt.bar(x, recall, width, label='Recall', color='lightgreen')
    bars3 = plt.bar(x + width, f1, width, label='F1-Score', color='salmon')

    # Add labels on top of the bars
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            yval = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, round(yval, 2), ha='center', va='bottom', fontsize=8)

    plt.xticks(x, emotions)
    plt.ylabel('Score')
    plt.ylim(0,1.1) # Adjusted y-limit to make space for labels
    plt.title(f'Performance Metrics by Emotion - {model_name}')
    plt.legend()
    plt.show()

# Plot for Logistic Regression
plot_metrics(y_test, lr_preds, "Logistic Regression")

# Plot for SVM
plot_metrics(y_test, svm_preds, "SVM")

In [ ]:
# Function for testing custom user input
def test_emotion_input():
    # Ask user to enter text
    user_text = input("Enter a sentence or social media comment: ")

    # Transform input using TF-IDF
    user_tfidf = tfidf.transform([user_text])

    # Logistic Regression prediction with probabilities
    lr_pred = lr_model.predict(user_tfidf)[0]
    lr_prob = lr_model.predict_proba(user_tfidf)[0]
    lr_confidence = max(lr_prob) * 100

    print("\n--- Logistic Regression ---")
    print(f"Predicted Emotion: {lr_pred}")
    print(f"Confidence: {lr_confidence:.2f}%")
    print(f"Probability Distribution: {dict(zip(lr_model.classes_, lr_prob.round(3)))}")

    # SVM prediction (with probability enabled during training)
    svm_pred = svm_model.predict(user_tfidf)[0]
    if hasattr(svm_model, "predict_proba"):
        svm_prob = svm_model.predict_proba(user_tfidf)[0]
        svm_confidence = max(svm_prob) * 100
        print("\n--- SVM ---")
        print(f"Predicted Emotion: {svm_pred}")
        print(f"Confidence: {svm_confidence:.2f}%")
        print(f"Probability Distribution: {dict(zip(svm_model.classes_, svm_prob.round(3)))}")
    else:
        print("\n--- SVM ---")
        print(f"Predicted Emotion: {svm_pred} (probability not enabled)")

# Run the function
test_emotion_input()


# TESTING COPY TEXT

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from torch.nn.functional import softmax
import torch

# --- Step 1: Load tokenizer and fine-tuned BERT model ---
# Replace 'my-bert-model' with your fine-tuned model path or Hugging Face model name
BERT_MODEL_NAME = "bert-base-multilingual-cased" # Using the base model for demonstration
tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)
model = BertForSequenceClassification.from_pretrained(BERT_MODEL_NAME)

# --- Step 2: Prediction function ---
def predict_bullying(text, threshold=0.7):
    # Tokenize input
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    # Get model outputs
    outputs = model(**inputs)
    probs = softmax(outputs.logits, dim=1)

    # Extract probabilities
    bully_prob = probs[0][1].item()
    not_bully_prob = probs[0][0].item()

    # Apply threshold
    if bully_prob >= threshold:
        label = "🔴Bully"
        confidence = bully_prob * 100
    else:
        label = "🟢Not Bully"
        confidence = not_bully_prob * 100

    return f"Classification: {label} ({confidence:.2f}%)"

# --- Step 3: Test ---
while True:
    user_input = input("Enter a sentence (or type 'exit' to quit): ")
    if user_input.lower() == "exit":
        break
    print(predict_bullying(user_input))

LOGISTIC REGRESSION TESTING

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def predict_bullying(text, threshold=0.7):
    # Transform input text using the fitted TF-IDF vectorizer
    X_input = tfidf.transform([text])

    # Get prediction probabilities
    probs = lr.predict_proba(X_input)[0]

    # Probability of Bully class (1) and Not Bully class (0)
    bully_prob = probs[1]
    not_bully_prob = probs[0]

    # Apply threshold
    # If the probability of "bully" is greater than or equal to the threshold,
    # classify the text as Bully and show the confidence percentage.
    if bully_prob >= threshold:
        label = "🔴Bully"
        confidence = bully_prob * 100
    else:
    # Otherwise, classify the text as Not Bully and show the confidence percentage.
        label = "🟢Not Bully"
        confidence = not_bully_prob * 100

    return f"Classification: {label} ({confidence:.2f}%)"

# --- Interactive Testing Loop ---
while True:
    user_input = input("Enter a sentence (or type 'exit' to quit): ")
    if user_input.lower() == "exit":
        break
    print(predict_bullying(user_input))


In [ ]:
from sklearn.svm import SVC

print("\nSVM...")

svm = SVC(
    kernel='linear',
    C=1.0,
    random_state=42,
    probability=True   # ⭐ REQUIRED for predict_proba()
)

svm.fit(X_tr, y_tr)

y_pred_svm = svm.predict(X_te)

evaluate_model("SVM", y_te, y_pred_svm)
plot_confusion_matrix("SVM", y_te, y_pred_svm)

In [ ]:
# @title
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC

# IMPORTANT:
# You must train this model first:
# svm = SVC(kernel='linear', probability=True)

def predict_bullying(text, threshold=0.7):
    # Transform input text using the fitted TF-IDF vectorizer
    X_input = tfidf.transform([text])

    # Get prediction probabilities
    probs = svm.predict_proba(X_input)[0]

    # Probability of Bully class (1) and Not Bully class (0)
    bully_prob = probs[1]
    not_bully_prob = probs[0]

    # Apply threshold
    if bully_prob >= threshold:
        label = "🔴Bully"
        confidence = bully_prob * 100
    else:
        label = "🟢Not Bully"
        confidence = not_bully_prob * 100

    return f"Classification: {label} ({confidence:.2f}%)"


# --- Interactive Testing Loop ---
while True:
    user_input = input("Enter a sentence (or type 'exit' to quit): ")
    if user_input.lower() == "exit":
        break
    print(predict_bullying(user_input))

#BULLY AND EMOTION INTEGRATE TESTING

In [ ]:
import numpy as np

def test_text_input():
    while True:
        text = input("Enter a sentence (or type 'exit' to quit): ")
        if text.lower() == "exit":
            break

        # Preprocess the user input
        cleaned_text, _ = preprocess_text(text)
        if not cleaned_text: # Handle cases where preprocessing results in empty string
            print("Input text became empty after preprocessing. Please try another input.")
            continue

        # Transform input using the correct TF-IDF vectorizer for bullying detection
        X_input_bully = bully_tfidf_vectorizer.transform([cleaned_text])

        # Step 1: Bullying detection
        bully_pred = lr.predict(X_input_bully)[0]   # your bully model
        bully_prob = lr.predict_proba(X_input_bully)[0][1]

        if bully_pred == 1:
            # Transform input using the correct TF-IDF vectorizer for emotion classification
            X_input_emotion = emotion_tfidf_vectorizer.transform([cleaned_text])

            # Step 2: Emotion classification (only if bullying detected)
            emotion_pred = lr_model.predict(X_input_emotion)[0]   # your emotion model
            emotion_probs = lr_model.predict_proba(X_input_emotion)[0]
            emotion_conf = np.max(emotion_probs)

            print(f"\n🚨 Bullying detected! (Confidence: {bully_prob:.2f})")
            print(f"Emotion: {emotion_pred} (Confidence: {emotion_conf:.2f})\n")
        else:
            print(f"\n✅ Not bullying (Confidence: {1-bully_prob:.2f})\n")

# Run the loop
test_text_input()

#Threshold Tuning Code

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Get probabilities for the positive class (bullying)
y_scores = lr.predict_proba(X_te)[:, 1]

# Define thresholds to test
thresholds = [0.5, 0.6, 0.7, 0.8]

print("Threshold tuning results:\n")
for t in thresholds:
    y_pred_custom = (y_scores >= t).astype(int)

    precision = precision_score(y_te, y_pred_custom)
    recall = recall_score(y_te, y_pred_custom)
    f1 = f1_score(y_te, y_pred_custom)

    print(f"Threshold = {t}")
    print(f"  Precision: {precision:.2f}")
    print(f"  Recall:    {recall:.2f}")
    print(f"  F1-score:  {f1:.2f}\n")


In [ ]:
from sklearn.calibration import CalibratedClassifierCV

calibrated_lr = CalibratedClassifierCV(estimator=lr, method='isotonic', cv=5)
calibrated_lr.fit(X_tr, y_tr)

# Use calibrated probabilities
y_scores_calibrated = calibrated_lr.predict_proba(X_te)[:, 1]